# Derivations for Numerical Solution of Geodesics for the Kerr Metric

In [ ]:
# Load sympy
from sympy import *

# Setup symbolic variables
t, r, theta, phi, GM, a = symbols('t r theta phi GM, a', real=True)

Create and fill the coordinates $x^\mu$, the Schwarzschild Metric Tensor $g_{\mu \nu}$, and its inverse $g^{\mu \nu}$

Spherical coordinates are used: $x^\mu = \begin{pmatrix} t \\ r \\ \theta \\ \phi \end{pmatrix}$

In [2]:
x = MutableDenseNDimArray.zeros(4)
x[0] = t
x[1] = r
x[2] = theta
x[3] = phi

x

[t, r, theta, phi]

In [3]:
g = MutableDenseNDimArray.zeros(4, 4)

Delta = r**2 - 2 * GM * r + a**2
rhosq = r**2 + a**2 * cos(theta)**2

g[0, 0] = -(1 - 2 * GM * r / rhosq)
g[0, 3] = -2 * GM * a * r * sin(theta)**2 / rhosq
g[3, 0] = g[0, 3]
g[1, 1] = rhosq / Delta
g[2, 2] = rhosq
g[3, 3] = sin(theta)**2 / rhosq * ((r**2 + a**2)**2 - a**2 * Delta * sin(theta)**2)

g

[[2*GM*r/(a**2*cos(theta)**2 + r**2) - 1, 0, 0, -2*GM*a*r*sin(theta)**2/(a**2*cos(theta)**2 + r**2)], [0, (a**2*cos(theta)**2 + r**2)/(-2*GM*r + a**2 + r**2), 0, 0], [0, 0, a**2*cos(theta)**2 + r**2, 0], [-2*GM*a*r*sin(theta)**2/(a**2*cos(theta)**2 + r**2), 0, 0, (-a**2*(-2*GM*r + a**2 + r**2)*sin(theta)**2 + (a**2 + r**2)**2)*sin(theta)**2/(a**2*cos(theta)**2 + r**2)]]

In [4]:
g_inv = MutableDenseNDimArray.zeros(4, 4)
for mu in range(4):
    g_inv[mu, mu] = 1 / g[mu, mu]

g_inv

[[1/(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1), 0, 0, 0], [0, (-2*GM*r + a**2 + r**2)/(a**2*cos(theta)**2 + r**2), 0, 0], [0, 0, 1/(a**2*cos(theta)**2 + r**2), 0], [0, 0, 0, (a**2*cos(theta)**2 + r**2)/((-a**2*(-2*GM*r + a**2 + r**2)*sin(theta)**2 + (a**2 + r**2)**2)*sin(theta)**2)]]

Generate the Christoffel Symbols $\Gamma^\lambda_{\mu\nu}$

In [5]:
christ_symbs = MutableDenseNDimArray.zeros(4, 4, 4)

# The three indexes of the Christoffel symbols
for lamb in range(4):
    for mu in range(4):
        for nu in range(4):

            # The dummy index for the summation
            for sigma in range(4):
                christ_symbs[lamb, mu, nu] += Rational(1, 2) * g_inv[lamb, sigma] * \
                    (diff(g[nu, sigma], x[mu]) + diff(g[sigma, mu], x[nu]) - diff(g[mu, nu], x[sigma]))

christ_symbs

[[[0, (-4*GM*r**2/(a**2*cos(theta)**2 + r**2)**2 + 2*GM/(a**2*cos(theta)**2 + r**2))/(2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1)), 2*GM*a**2*r*sin(theta)*cos(theta)/((a**2*cos(theta)**2 + r**2)**2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1)), 0], [(-4*GM*r**2/(a**2*cos(theta)**2 + r**2)**2 + 2*GM/(a**2*cos(theta)**2 + r**2))/(2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1)), 0, 0, (4*GM*a*r**2*sin(theta)**2/(a**2*cos(theta)**2 + r**2)**2 - 2*GM*a*sin(theta)**2/(a**2*cos(theta)**2 + r**2))/(2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1))], [2*GM*a**2*r*sin(theta)*cos(theta)/((a**2*cos(theta)**2 + r**2)**2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1)), 0, 0, (-4*GM*a**3*r*sin(theta)**3*cos(theta)/(a**2*cos(theta)**2 + r**2)**2 - 4*GM*a*r*sin(theta)*cos(theta)/(a**2*cos(theta)**2 + r**2))/(2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1))], [0, (4*GM*a*r**2*sin(theta)**2/(a**2*cos(theta)**2 + r**2)**2 - 2*GM*a*sin(theta)**2/(a**2*cos(theta)**2 + r**2))/(2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1)), (-4*GM*a**3*

Now generate equations of motion $\frac{\text{d}^2 x^\mu}{\text{d}\tau^2}$

In [6]:
# First define symbols for the derivative of the coordinates
dt, dr, dtheta, dphi = symbols('tdot rdot thetadot phidot', real=True)

dx = MutableDenseNDimArray.zeros(4)
dx[0] = dt
dx[1] = dr
dx[2] = dtheta
dx[3] = dphi

dx

[tdot, rdot, thetadot, phidot]

In [7]:
ddx = MutableDenseNDimArray.zeros(4)

# The index of the second derivative coordinate
for mu in range(4):
    
    # Summations over the chrstoffel symbols and first derivatives
    for rho in range(4):
        for sigma in range(4):
            ddx[mu] -= christ_symbs[mu, rho, sigma] * dx[rho] * dx[sigma]

ddx = simplify(ddx)
ddx

[2*GM*(-a**2*r*tdot*thetadot*sin(2*theta) + a*phidot*r*thetadot*(a**2 + r**2)*sin(2*theta) - a*phidot*rdot*(-a**2*cos(theta)**2 + r**2)*sin(theta)**2 + rdot*tdot*(-a**2*cos(theta)**2 + r**2))/((a**2*cos(theta)**2 + r**2)*(2*GM*r - a**2*cos(theta)**2 - r**2)), (rdot**2*(a**2*cos(theta)**2 + r**2)**2*(-r*(-2*GM*r + a**2 + r**2) + (-GM + r)*(a**2*cos(theta)**2 + r**2)) + thetadot*(a**2*cos(theta)**2 + r**2)**2*(a**2*rdot*sin(2*theta) + r*thetadot*(-2*GM*r + a**2 + r**2))*(-2*GM*r + a**2 + r**2) + (-2*GM*r + a**2 + r**2)**2*(2*GM*a*phidot*tdot*(-a**2*cos(theta)**2 + r**2)*sin(theta)**2 - GM*tdot**2*(-a**2*cos(theta)**2 + r**2) + phidot**2*(r*(a**2*(-2*GM*r + a**2 + r**2)*sin(theta)**2 - (a**2 + r**2)**2) + (a**2*cos(theta)**2 + r**2)*(a**2*(GM - r)*sin(theta)**2 + 2*r*(a**2 + r**2)))*sin(theta)**2))/((a**2*cos(theta)**2 + r**2)**3*(-2*GM*r + a**2 + r**2)), (GM*a**2*r*tdot**2*(-2*GM*r + a**2 + r**2)*sin(2*theta) - a**2*rdot**2*(a**2*cos(2*theta) + a**2 + 2*r**2)**2*sin(2*theta)/8 + phidot*(

Convert the resulting equation to python code

In [8]:
# Print function code
from sympy.printing.numpy import NumPyPrinter
NumPyPrinter().doprint(ddx)

'numpy.array([2*GM*(-a**2*r*tdot*thetadot*numpy.sin(2*theta) + a*phidot*r*thetadot*(a**2 + r**2)*numpy.sin(2*theta) - a*phidot*rdot*(-a**2*numpy.cos(theta)**2 + r**2)*numpy.sin(theta)**2 + rdot*tdot*(-a**2*numpy.cos(theta)**2 + r**2))/((a**2*numpy.cos(theta)**2 + r**2)*(2*GM*r - a**2*numpy.cos(theta)**2 - r**2)), (rdot**2*(a**2*numpy.cos(theta)**2 + r**2)**2*(-r*(-2*GM*r + a**2 + r**2) + (-GM + r)*(a**2*numpy.cos(theta)**2 + r**2)) + thetadot*(a**2*numpy.cos(theta)**2 + r**2)**2*(a**2*rdot*numpy.sin(2*theta) + r*thetadot*(-2*GM*r + a**2 + r**2))*(-2*GM*r + a**2 + r**2) + (-2*GM*r + a**2 + r**2)**2*(2*GM*a*phidot*tdot*(-a**2*numpy.cos(theta)**2 + r**2)*numpy.sin(theta)**2 - GM*tdot**2*(-a**2*numpy.cos(theta)**2 + r**2) + phidot**2*(r*(a**2*(-2*GM*r + a**2 + r**2)*numpy.sin(theta)**2 - (a**2 + r**2)**2) + (a**2*numpy.cos(theta)**2 + r**2)*(a**2*(GM - r)*numpy.sin(theta)**2 + 2*r*(a**2 + r**2)))*numpy.sin(theta)**2))/((a**2*numpy.cos(theta)**2 + r**2)**3*(-2*GM*r + a**2 + r**2)), (GM*a**2

In [9]:
# Print used variables
ddx.free_symbols

{GM, a, phidot, r, rdot, tdot, theta, thetadot}

Now add some coordinate transformation code

In [10]:
cart = MutableDenseNDimArray.zeros(4)

cart[0] = t
cart[1] = r * sin(theta) * cos(phi)
cart[2] = r * sin(theta) * sin(phi)
cart[3] = r * cos(theta)

cart

[t, r*sin(theta)*cos(phi), r*sin(phi)*sin(theta), r*cos(theta)]

In [11]:
# Create coordinate transformation matrix: spherical to cartesian
dcart_dsph = Matrix.zeros(4, 4)

for mu_ in range(4):
    for mu in range(4):
        dcart_dsph[mu_, mu] = diff(cart[mu_], x[mu])

# Take inverse: cartesian to spherical
dsph_dcart = simplify(dcart_dsph.inv())

In [12]:
dcart_dsph

Matrix([
[1,                   0,                     0,                      0],
[0, sin(theta)*cos(phi), r*cos(phi)*cos(theta), -r*sin(phi)*sin(theta)],
[0, sin(phi)*sin(theta), r*sin(phi)*cos(theta),  r*sin(theta)*cos(phi)],
[0,          cos(theta),         -r*sin(theta),                      0]])

In [13]:
dsph_dcart

Matrix([
[1,                        0,                       0,             0],
[0,      sin(theta)*cos(phi),     sin(phi)*sin(theta),    cos(theta)],
[0,    cos(phi)*cos(theta)/r,   sin(phi)*cos(theta)/r, -sin(theta)/r],
[0, -sin(phi)/(r*sin(theta)), cos(phi)/(r*sin(theta)),             0]])

Spherical velocity to cartesian velocity

In [14]:
dcart = MutableDenseNDimArray.zeros(4)

for mu_ in range(4):
    for mu in range(4):
        dcart[mu_] += dcart_dsph[mu_, mu] * dx[mu]

dcart

[tdot, -phidot*r*sin(phi)*sin(theta) + r*thetadot*cos(phi)*cos(theta) + rdot*sin(theta)*cos(phi), phidot*r*sin(theta)*cos(phi) + r*thetadot*sin(phi)*cos(theta) + rdot*sin(phi)*sin(theta), -r*thetadot*sin(theta) + rdot*cos(theta)]

In [15]:
NumPyPrinter().doprint(dcart)

'numpy.array([tdot, -phidot*r*numpy.sin(phi)*numpy.sin(theta) + r*thetadot*numpy.cos(phi)*numpy.cos(theta) + rdot*numpy.sin(theta)*numpy.cos(phi), phidot*r*numpy.sin(theta)*numpy.cos(phi) + r*thetadot*numpy.sin(phi)*numpy.cos(theta) + rdot*numpy.sin(phi)*numpy.sin(theta), -r*thetadot*numpy.sin(theta) + rdot*numpy.cos(theta)])'

In [16]:
dcart.free_symbols

{phi, phidot, r, rdot, tdot, theta, thetadot}

Cartesian velocity to spherical velocity

In [17]:
dt_cart, dx_cart, dy_cart, dz_cart = symbols('tdot_cart xdot_cart ydot_cart zdot_cart')
dcart_symb = MutableDenseNDimArray.zeros(4)
dcart_symb[0] = dt_cart
dcart_symb[1] = dx_cart
dcart_symb[2] = dy_cart
dcart_symb[3] = dz_cart

dx_conv = MutableDenseNDimArray.zeros(4)

for mu in range(4):
    for mu_ in range(4):
        dx_conv[mu] += dsph_dcart[mu, mu_] * dcart_symb[mu_]

dx_conv = simplify(dx_conv)
dx_conv

[tdot_cart, xdot_cart*sin(theta)*cos(phi) + ydot_cart*sin(phi)*sin(theta) + zdot_cart*cos(theta), (xdot_cart*cos(phi)*cos(theta) + ydot_cart*sin(phi)*cos(theta) - zdot_cart*sin(theta))/r, (-xdot_cart*sin(phi) + ydot_cart*cos(phi))/(r*sin(theta))]

In [18]:
NumPyPrinter().doprint(dx_conv)

'numpy.array([tdot_cart, xdot_cart*numpy.sin(theta)*numpy.cos(phi) + ydot_cart*numpy.sin(phi)*numpy.sin(theta) + zdot_cart*numpy.cos(theta), (xdot_cart*numpy.cos(phi)*numpy.cos(theta) + ydot_cart*numpy.sin(phi)*numpy.cos(theta) - zdot_cart*numpy.sin(theta))/r, (-xdot_cart*numpy.sin(phi) + ydot_cart*numpy.cos(phi))/(r*numpy.sin(theta))])'

In [19]:
dx_conv.free_symbols

{phi, r, tdot_cart, theta, xdot_cart, ydot_cart, zdot_cart}

Make equation to enforce that initial four-velocity satisfies $U_\mu U^\mu = -1$

In [20]:
# Compute norm
dx_norm = 0

for mu in range(4):
    for nu in range(4):
        dx_norm += g[mu, nu] * dx[nu] * dx[mu]

dx_norm

-4*GM*a*phidot*r*tdot*sin(theta)**2/(a**2*cos(theta)**2 + r**2) + phidot**2*(-a**2*(-2*GM*r + a**2 + r**2)*sin(theta)**2 + (a**2 + r**2)**2)*sin(theta)**2/(a**2*cos(theta)**2 + r**2) + rdot**2*(a**2*cos(theta)**2 + r**2)/(-2*GM*r + a**2 + r**2) + tdot**2*(2*GM*r/(a**2*cos(theta)**2 + r**2) - 1) + thetadot**2*(a**2*cos(theta)**2 + r**2)

In [21]:
# Should be the positive of the two solutions
dt_sol = solve(Eq(dx_norm, -1), dt)[0]
dt_sol

(-2*GM*a*phidot*r*(4*GM**2*r**2 - 2*GM*a**2*r*cos(theta)**2 - 2*GM*a**2*r - 4*GM*r**3 + a**4*cos(theta)**2 + a**2*r**2*cos(theta)**2 + a**2*r**2 + r**4)*sin(theta)**2 + sqrt((-2*GM*r + a**2 + r**2)*(4*GM**2*a**4*phidot**2*r**2*sin(theta)**6 - 8*GM**2*a**4*phidot**2*r**2*sin(theta)**4 + 4*GM**2*a**4*phidot**2*r**2*sin(theta)**2 + 4*GM**2*a**4*r**2*thetadot**2*sin(theta)**4 - 8*GM**2*a**4*r**2*thetadot**2*sin(theta)**2 + 4*GM**2*a**4*r**2*thetadot**2 - 8*GM**2*a**2*phidot**2*r**4*sin(theta)**4 + 8*GM**2*a**2*phidot**2*r**4*sin(theta)**2 - 8*GM**2*a**2*r**4*thetadot**2*sin(theta)**2 + 8*GM**2*a**2*r**4*thetadot**2 - 4*GM**2*a**2*r**2*sin(theta)**2 + 4*GM**2*a**2*r**2 + 4*GM**2*phidot**2*r**6*sin(theta)**2 + 4*GM**2*r**6*thetadot**2 + 4*GM**2*r**4 - 4*GM*a**6*phidot**2*r*sin(theta)**6 + 8*GM*a**6*phidot**2*r*sin(theta)**4 - 4*GM*a**6*phidot**2*r*sin(theta)**2 - 2*GM*a**6*r*thetadot**2*sin(theta)**4 + 4*GM*a**6*r*thetadot**2*sin(theta)**2 - 2*GM*a**6*r*thetadot**2*cos(theta)**6 - 2*GM*a**6*

In [22]:
NumPyPrinter().doprint(dt_sol)

'(-2*GM*a*phidot*r*(4*GM**2*r**2 - 2*GM*a**2*r*numpy.cos(theta)**2 - 2*GM*a**2*r - 4*GM*r**3 + a**4*numpy.cos(theta)**2 + a**2*r**2*numpy.cos(theta)**2 + a**2*r**2 + r**4)*numpy.sin(theta)**2 + numpy.sqrt((-2*GM*r + a**2 + r**2)*(4*GM**2*a**4*phidot**2*r**2*numpy.sin(theta)**6 - 8*GM**2*a**4*phidot**2*r**2*numpy.sin(theta)**4 + 4*GM**2*a**4*phidot**2*r**2*numpy.sin(theta)**2 + 4*GM**2*a**4*r**2*thetadot**2*numpy.sin(theta)**4 - 8*GM**2*a**4*r**2*thetadot**2*numpy.sin(theta)**2 + 4*GM**2*a**4*r**2*thetadot**2 - 8*GM**2*a**2*phidot**2*r**4*numpy.sin(theta)**4 + 8*GM**2*a**2*phidot**2*r**4*numpy.sin(theta)**2 - 8*GM**2*a**2*r**4*thetadot**2*numpy.sin(theta)**2 + 8*GM**2*a**2*r**4*thetadot**2 - 4*GM**2*a**2*r**2*numpy.sin(theta)**2 + 4*GM**2*a**2*r**2 + 4*GM**2*phidot**2*r**6*numpy.sin(theta)**2 + 4*GM**2*r**6*thetadot**2 + 4*GM**2*r**4 - 4*GM*a**6*phidot**2*r*numpy.sin(theta)**6 + 8*GM*a**6*phidot**2*r*numpy.sin(theta)**4 - 4*GM*a**6*phidot**2*r*numpy.sin(theta)**2 - 2*GM*a**6*r*thetadot*

In [23]:
dt_sol.free_symbols

{GM, a, phidot, r, rdot, theta, thetadot}